# 데이터 전처리

In [3]:
import pandas as pd
import numpy as np

clinical = pd.read_csv('../clinical/clinical.tsv', sep='\t')
print(f"Shape: {clinical.shape}")
clinical.head()

Shape: (1010, 210)


,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,...,treatments.treatment_duration,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type
0,TCGA-BRCA,0130d616-885e-4a6c-9d03-2f17dd692a05,Informed Consent,36,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-D8-A1XY,...,'--,'--,'--,'--,0919b938-b1d4-495c-9949-53476fd17530,Adjuvant,no,'--,'--,"Radiation Therapy, NOS"
1,TCGA-BRCA,0130d616-885e-4a6c-9d03-2f17dd692a05,Informed Consent,36,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-D8-A1XY,...,'--,'--,'--,'--,694aa917-d197-4567-a275-cdbe36f950d4,First-Line Therapy,yes,'--,'--,"Surgery, NOS"
2,TCGA-BRCA,0130d616-885e-4a6c-9d03-2f17dd692a05,Informed Consent,36,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-D8-A1XY,...,'--,'--,'--,'--,ee020e6d-56e9-5b80-a3cb-4c2384734daf,Adjuvant,yes,Treatment Ongoing,'--,Chemotherapy
3,TCGA-BRCA,02bed00f-bef7-4fb7-b243-540354990e45,Informed Consent,0,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-A2-A0T1,...,'--,'--,'--,'--,079b674f-ec4a-4eab-9c5c-f1aa77d4b319,Adjuvant,yes,Treatment Ongoing,'--,Targeted Molecular Therapy
4,TCGA-BRCA,02bed00f-bef7-4fb7-b243-540354990e45,Informed Consent,0,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-A2-A0T1,...,'--,'--,'--,'--,59972af7-d937-499c-b8ef-0f2f0106575f,Adjuvant,yes,'--,'--,Chemotherapy


In [4]:
patient_cols = [col for col in clinical.columns if 'case' in col.lower() or 'submitter' in col.lower()]
print(patient_cols[:10])

['cases.case_id', 'cases.consent_type', 'cases.days_to_consent', 'cases.days_to_lost_to_followup', 'cases.disease_type', 'cases.index_date', 'cases.lost_to_followup', 'cases.primary_site', 'cases.submitter_id', 'demographic.submitter_id']


In [5]:
print(f"case_id 고유값: {clinical['cases.case_id'].nunique()}")
print(f"submitter_id 고유값: {clinical['cases.submitter_id'].nunique()}")

# submitter_id 예시
print(f"\nsubmitter_id 예시:")
print(clinical['cases.submitter_id'].unique()[:5])

case_id 고유값: 194
submitter_id 고유값: 194

submitter_id 예시:
['TCGA-D8-A1XY' 'TCGA-A2-A0T1' 'TCGA-D8-A1JC' 'TCGA-C8-A137'
 'TCGA-B6-A0I6']


In [6]:
unique_patients = clinical['cases.case_id'].nunique()
print(f"고유 환자 수: {unique_patients}")

records_per_patient = clinical['cases.case_id'].value_counts()
print(f"\n환자당 레코드 수 분포:")
print(records_per_patient.describe())
print(f"\n예시 (상위 5명):")
print(records_per_patient.head())

고유 환자 수: 194

환자당 레코드 수 분포:
count    194.000000
mean       5.206186
std        3.715081
min        1.000000
25%        3.000000
50%        5.000000
75%        6.000000
max       40.000000
Name: count, dtype: float64

예시 (상위 5명):
cases.case_id
c07b122e-ac50-4db2-add2-5617a5d0e976    40
deba32e4-0e68-4711-941b-3b63bd965afb    21
4a831893-f4ca-4357-a756-b5e954e35dd7    19
cc0902fd-a2fc-4327-929e-d4219e32e1d7    14
a8f5c479-8685-4e2d-bb60-63f1cc651083    13
Name: count, dtype: int64


환자당 평균 5개 레코드, 최대 40개 레코드 

In [7]:
survival_cols = [
    'cases.submitter_id',
    'demographic.days_to_death',
    'demographic.vital_status', 
    'diagnoses.days_to_last_follow_up',
    'demographic.age_at_index',
    'demographic.race',
    'demographic.gender',
    'diagnoses.ajcc_pathologic_stage',
    'diagnoses.tumor_grade',
    'diagnoses.primary_diagnosis'
]

df_survival = clinical[survival_cols].copy()
print(df_survival.head(3))
print(df_survival.isnull().sum())

  cases.submitter_id demographic.days_to_death demographic.vital_status  \
0       TCGA-D8-A1XY                       '--                    Alive   
1       TCGA-D8-A1XY                       '--                    Alive   
2       TCGA-D8-A1XY                       '--                    Alive   

  diagnoses.days_to_last_follow_up  demographic.age_at_index demographic.race  \
0                            503.0                        74            white   
1                            503.0                        74            white   
2                            503.0                        74            white   

  demographic.gender diagnoses.ajcc_pathologic_stage diagnoses.tumor_grade  \
0             female                       Stage IIA                   '--   
1             female                       Stage IIA                   '--   
2             female                       Stage IIA                   '--   

        diagnoses.primary_diagnosis  
0  Infiltrating duct ca

In [8]:
df_survival = df_survival.mask(df_survival == "'--")
df_survival.isnull().sum() / len(df_survival) * 100

cases.submitter_id                    0.000000
demographic.days_to_death            81.881188
demographic.vital_status              0.000000
diagnoses.days_to_last_follow_up     10.198020
demographic.age_at_index              0.000000
demographic.race                      0.000000
demographic.gender                    0.000000
diagnoses.ajcc_pathologic_stage      10.495050
diagnoses.tumor_grade               100.000000
diagnoses.primary_diagnosis           0.000000
dtype: float64

In [9]:
df_patient = df_survival.groupby('cases.submitter_id').first().reset_index()
df_patient = df_patient.drop(columns=['diagnoses.tumor_grade'])
print(df_patient['demographic.vital_status'].value_counts())

demographic.vital_status
Alive    168
Dead      26
Name: count, dtype: int64


생존 시간(time)과 이벤트(event) 변수 생성

- 사망자: days_to_death 사용
- 생존자: days_to_last_follow_up 사용

In [10]:
df_patient['survival_time'] = df_patient['demographic.days_to_death'].fillna(
    df_patient['diagnoses.days_to_last_follow_up']
)
df_patient['event'] = (df_patient['demographic.vital_status'] == 'Dead').astype(int)

print(df_patient['survival_time'].describe())
print('\n')
print(df_patient['event'].value_counts())
print('\n')
print(df_patient[['cases.submitter_id', 'demographic.vital_status', 
                  'survival_time', 'event']].head(10))

count      194
unique     176
top       10.0
freq         6
Name: survival_time, dtype: object


event
0    168
1     26
Name: count, dtype: int64


  cases.submitter_id demographic.vital_status survival_time  event
0       TCGA-3C-AALI                    Alive        4005.0      0
1       TCGA-5L-AAT0                    Alive        1477.0      0
2       TCGA-A1-A0SE                    Alive        1321.0      0
3       TCGA-A2-A04T                    Alive        2246.0      0
4       TCGA-A2-A04U                    Alive        2654.0      0
5       TCGA-A2-A0CY                    Alive        1673.0      0
6       TCGA-A2-A0CZ                    Alive        1616.0      0
7       TCGA-A2-A0EM                    Alive        3094.0      0
8       TCGA-A2-A0ST                    Alive        3017.0      0
9       TCGA-A2-A0T1                    Alive         521.0      0


In [11]:
df_patient['survival_time'].dtype

dtype('O')

In [12]:
df_patient['survival_time'] = pd.to_numeric(df_patient['survival_time'])
df_patient['survival_time'].dtype

dtype('float64')

In [13]:
print(df_patient['survival_time'].describe())
print(f"\nsurvival_time 결측: {df_patient['survival_time'].isnull().sum()}개")

count     194.000000
mean     1189.804124
std      1118.458985
min         0.000000
25%       449.500000
50%       825.000000
75%      1619.000000
max      6593.000000
Name: survival_time, dtype: float64

survival_time 결측: 0개


생존 데이터 준비 완료  
Cox 모델에 필요한 형태로 변환

In [14]:
from sksurv.util import Surv

y = Surv.from_arrays(
    event=df_patient['event'].astype(bool),
    time=df_patient['survival_time']
)

print("생존 데이터 형태:", y.dtype)
print("샘플 5개:")
print(y[:5])

생존 데이터 형태: [('event', '?'), ('time', '<f8')]
샘플 5개:
[(False, 4005.) (False, 1477.) (False, 1321.) (False, 2246.)
 (False, 2654.)]


Cox 모델

In [15]:
from sklearn.preprocessing import LabelEncoder
print("Stage 분포:")
print(df_patient['diagnoses.ajcc_pathologic_stage'].value_counts())

Stage 분포:
diagnoses.ajcc_pathologic_stage
Stage IIA     67
Stage IIB     47
Stage IIIA    23
Stage IIIC    17
Stage I       15
Stage IA      13
Stage X        3
Stage IIIB     2
Stage III      2
Stage II       2
Stage IV       1
Stage IB       1
Name: count, dtype: int64


In [16]:
stage_map = {
    'Stage I': 1, 'Stage IA': 1, 'Stage IB': 1,
    'Stage II': 2, 'Stage IIA': 2, 'Stage IIB': 2,
    'Stage III': 3, 'Stage IIIA': 3, 'Stage IIIB': 3, 'Stage IIIC': 3,
    'Stage IV': 4
}
df_patient['stage_numeric'] = df_patient['diagnoses.ajcc_pathologic_stage'].map(stage_map)

print("\nStage 변환 결과:")
print(df_patient[['diagnoses.ajcc_pathologic_stage', 'stage_numeric']].head(10))


Stage 변환 결과:
  diagnoses.ajcc_pathologic_stage  stage_numeric
0                       Stage IIB            2.0
1                       Stage IIA            2.0
2                         Stage I            1.0
3                       Stage IIA            2.0
4                       Stage IIA            2.0
5                       Stage IIB            2.0
6                       Stage IIA            2.0
7                        Stage IA            1.0
8                       Stage IIA            2.0
9                      Stage IIIC            3.0


범주형 변수 변환

In [17]:
# Gender
df_patient['gender_numeric'] = (df_patient['demographic.gender'] == 'male').astype(int)

# Race - One-hot encoding
df_patient = pd.get_dummies(df_patient, columns=['demographic.race'], prefix='race', drop_first=True)

print("\nPrimary diagnosis 분포:")
print(df_patient['diagnoses.primary_diagnosis'].value_counts())


Primary diagnosis 분포:
diagnoses.primary_diagnosis
Infiltrating duct carcinoma, NOS                         139
Lobular carcinoma, NOS                                    31
Not Reported                                               6
Infiltrating duct and lobular carcinoma                    5
Metaplastic carcinoma, NOS                                 3
Invasive micropapillary carcinoma                          2
Medullary carcinoma, NOS                                   2
Adenocarcinoma, NOS                                        1
Papillary carcinoma, NOS                                   1
Infiltrating duct mixed with other types of carcinoma      1
Tubular adenocarcinoma                                     1
Intraductal carcinoma, noninfiltrating, NOS                1
Pleomorphic carcinoma                                      1
Name: count, dtype: int64


- Infiltrating duct carcinoma (침윤성 관암):139명
- Lobular carcinoma (소엽암): 31명
- 나머지: 너무 적음

In [18]:
# Primary diagnosis를 3개 카테고리로 단순화
def simplify_diagnosis(dx):
    if pd.isna(dx) or dx == 'Not Reported':
        return 'Other'
    elif 'duct' in dx.lower():
        return 'Ductal'
    elif 'Lobular' in dx:
        return 'Lobular'
    else:
        return 'Other'

df_patient['diagnosis_simplified'] = df_patient['diagnoses.primary_diagnosis'].apply(simplify_diagnosis)

print("단순화된 진단:")
print(df_patient['diagnosis_simplified'].value_counts())

# One-hot encoding
df_patient = pd.get_dummies(df_patient, columns=['diagnosis_simplified'], prefix='dx', drop_first=True)

# 최종 feature 리스트
feature_cols = ['demographic.age_at_index', 'stage_numeric', 'gender_numeric']

# race와 diagnosis one-hot 컬럼 추가
race_cols = [col for col in df_patient.columns if col.startswith('race_')]
dx_cols = [col for col in df_patient.columns if col.startswith('dx_')]

feature_cols = feature_cols + race_cols + dx_cols

print(f"\n최종 사용 변수 ({len(feature_cols)}개):")
print(feature_cols)

단순화된 진단:
diagnosis_simplified
Ductal     146
Lobular     31
Other       17
Name: count, dtype: int64

최종 사용 변수 (9개):
['demographic.age_at_index', 'stage_numeric', 'gender_numeric', 'race_asian', 'race_black or african american', 'race_not reported', 'race_white', 'dx_Lobular', 'dx_Other']


# Cox 모델 (임상 데이터만)


In [19]:
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("변수별 결측치:")
print(df_patient[feature_cols + ['survival_time', 'event']].isnull().sum())

if df_patient[feature_cols].isnull().any().any():
    print("\n결측치 있는 환자:")
    print(df_patient[df_patient[feature_cols].isnull().any(axis=1)][
        ['cases.submitter_id'] + feature_cols
    ])

변수별 결측치:
demographic.age_at_index          0
stage_numeric                     4
gender_numeric                    0
race_asian                        0
race_black or african american    0
race_not reported                 0
race_white                        0
dx_Lobular                        0
dx_Other                          0
survival_time                     0
event                             0
dtype: int64

결측치 있는 환자:
   cases.submitter_id  demographic.age_at_index  stage_numeric  \
29       TCGA-A8-A099                        76            NaN   
30       TCGA-A8-A09A                        40            NaN   
60       TCGA-AR-A0TZ                        43            NaN   
76       TCGA-B6-A0WS                        58            NaN   

    gender_numeric  race_asian  race_black or african american  \
29               0       False                           False   
30               0       False                           False   
60               0       False           

In [20]:
df_model = df_patient[feature_cols + ['survival_time', 'event']].dropna()

X = df_model[feature_cols].values
y = Surv.from_arrays(
    event=df_model['event'].astype(bool),
    time=df_model['survival_time']
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=df_model['event'],  # 이벤트 비율 맞춰서 분할
    random_state=42
)

print(f"Train: {len(X_train)}명, 이벤트 {y_train['event'].sum()}개")
print(f"Test: {len(X_test)}명, 이벤트 {y_test['event'].sum()}개")


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

cox_model = CoxPHSurvivalAnalysis()
cox_model.fit(X_train_scaled, y_train)

train_score = cox_model.score(X_train_scaled, y_train)
test_score = cox_model.score(X_test_scaled, y_test)

print(f"\nTrain C-index: {train_score:.4f}")
print(f"Test C-index: {test_score:.4f}")

Train: 152명, 이벤트 19개
Test: 38명, 이벤트 5개

Train C-index: 0.7988
Test C-index: 0.6863


# WSI Feature 매칭


In [29]:
import os
import torch

feature_dir = '../CLAM/results/features/pt_files'
feature_files = os.listdir(feature_dir)

slide_vectors = []
patient_ids = []


for f in feature_files:
    path = os.path.join(feature_dir, f)
    # 1) 패치 feature 로드
    feats = torch.load(path)   # shape: [num_patches, 1024]
    # 2) 슬라이드 수준 벡터 생성 (mean pooling)
    slide_vec = feats.mean(dim=0).numpy()   # shape: (1024,)
    # 3) patient_id 추출 (TCGA-A2-A0T1)
    patient_id = '-'.join(f.split('-')[:3])
    
    slide_vectors.append(slide_vec)
    patient_ids.append(patient_id)
    
# 4) DataFrame 생성
df_slide = pd.DataFrame(slide_vectors)
df_slide.insert(0, 'cases.submitter_id', patient_ids)

print("슬라이드 기준 행 개수:", len(df_slide))
print("슬라이드 기준 유니크 환자 수:", df_slide['cases.submitter_id'].nunique())


# 5) 환자당 슬라이드 1장만 사용 (가장 처음 것 keep)
df_slide_unique = df_slide.drop_duplicates(subset='cases.submitter_id', keep='first')
print("한 환자당 1장만 남긴 후 행 개수:", len(df_slide_unique))
print("유니크 환자 수:", df_slide_unique['cases.submitter_id'].nunique())

슬라이드 기준 행 개수: 200
슬라이드 기준 유니크 환자 수: 194
한 환자당 1장만 남긴 후 행 개수: 194
유니크 환자 수: 194


In [30]:
cols_for_merge = ['cases.submitter_id'] + feature_cols + ['survival_time', 'event']
df_model_with_id = df_patient[cols_for_merge].dropna()
print("임상 쪽 준비된 환자 수:", len(df_model_with_id))
df_merged = df_model_with_id.merge(df_slide_unique, on='cases.submitter_id', how='inner')
print("최종 merge된 환자 수:", len(df_merged))


임상 쪽 준비된 환자 수: 190
최종 merge된 환자 수: 190


In [42]:
df_merged.head(3)

,cases.submitter_id,demographic.age_at_index,stage_numeric,gender_numeric,race_asian,race_black or african american,race_not reported,race_white,dx_Lobular,dx_Other,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,TCGA-3C-AALI,50,2.0,0,False,True,False,False,False,False,...,0.014503,0.015180,0.016066,0.006847,0.003364,0.015421,0.025987,0.012698,0.040273,0.035296
1,TCGA-5L-AAT0,42,2.0,0,False,False,False,True,False,True,...,0.018714,0.015436,0.017545,0.015764,0.001323,0.065393,0.062858,0.005488,0.033258,0.050850
2,TCGA-A1-A0SE,56,1.0,0,False,False,False,True,False,False,...,0.014127,0.015874,0.011378,0.010092,0.002376,0.017531,0.040533,0.014539,0.065636,0.034699


# 통합 모델: 임상 + 이미지


In [32]:
from sklearn.pipeline import Pipeline
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis

In [35]:
df = df_merged.copy()

# 1) 입력(X), 타겟(y) 분리
drop_cols = ['cases.submitter_id', 'survival_time', 'event']
X = df.drop(columns=drop_cols)
X.columns = X.columns.astype(str)


# sksurv용 structured array로 변환
y = np.array(
    [(bool(e), float(t)) for e, t in zip(df['event'], df['survival_time'])],
    dtype=[('event', '?'), ('time', '<f8')]
)

# 2) train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df['event']  # event 비율 유지
)

## 1. Cox Regression


In [36]:
cox = Pipeline([
    ("scaler", StandardScaler()),
    ("model", CoxPHSurvivalAnalysis())
])

cox.fit(X_train, y_train)

cox_train_c = cox.score(X_train, y_train)  # concordance index
cox_test_c = cox.score(X_test, y_test)

print(f"CoxPH   - train C-index: {cox_train_c:.3f}, test C-index: {cox_test_c:.3f}")

/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=3.31665e-21): result may not be accurate.
  delta = solve(
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=9.89226e-22): result may not be accurate.
  delta = solve(
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=1.18184e-21): result may not be accurate.
  delta = solve(
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=2.6985e-22): result may not be accurate.
  delta = solve(
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=1.17363e-23): result may not be accurate.
  delta = solve(
/

CoxPH   - train C-index: 1.000, test C-index: 0.667


/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sklearn/pipeline.py:663: ConvergenceWarning: Optimization did not converge: Maximum number of iterations has been exceeded.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:86: RuntimeWarning: invalid value encountered in divide
  y = np.cumsum(n_events / divisor)


### Cox L2 penalty 튜닝 (alpha 여러 값 테스트)

In [40]:
alphas = [0, 1e-4, 1e-3, 1e-2, 1e-1]

cox_results = []

for a in alphas:
    cox_l2 = Pipeline([
        ("scaler", StandardScaler()),
        ("model", CoxPHSurvivalAnalysis(alpha=a))
    ])
    cox_l2.fit(X_train, y_train)
    train_c = cox_l2.score(X_train, y_train)
    test_c = cox_l2.score(X_test, y_test)
    cox_results.append((a, train_c, test_c))
    print(f"alpha={a:>6}  train C={train_c:.3f},  test C={test_c:.3f}")

/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=3.31665e-21): result may not be accurate.
  delta = solve(
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=9.89226e-22): result may not be accurate.
  delta = solve(
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=1.18184e-21): result may not be accurate.
  delta = solve(
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=2.6985e-22): result may not be accurate.
  delta = solve(
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:492: LinAlgWarning: Ill-conditioned matrix (rcond=1.17363e-23): result may not be accurate.
  delta = solve(
/

alpha=     0  train C=1.000,  test C=0.667
alpha=0.0001  train C=1.000,  test C=0.647
alpha= 0.001  train C=1.000,  test C=0.647
alpha=  0.01  train C=1.000,  test C=0.647


/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:197: RuntimeWarning: overflow encountered in exp
  risk_set2 += np.exp(xw[k])


alpha=   0.1  train C=1.000,  test C=0.647


### PCA + Cox (이미지 차원 축소)

In [43]:
image_cols = [c for c in X.columns if c not in feature_cols]

Xc_train = X_train[feature_cols]
Xi_train = X_train[image_cols]
Xc_test  = X_test[feature_cols]
Xi_test  = X_test[image_cols]

from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
Xi_train_pca = pca.fit_transform(Xi_train)
Xi_test_pca  = pca.transform(Xi_test)

# DataFrame으로 다시 묶기 (컬럼명은 pca_0, pca_1, ... 로)
Xi_train_pca_df = pd.DataFrame(
    Xi_train_pca,
    index=X_train.index,
    columns=[f"img_pca_{i}" for i in range(Xi_train_pca.shape[1])]
)

Xi_test_pca_df = pd.DataFrame(
    Xi_test_pca,
    index=X_test.index,
    columns=[f"img_pca_{i}" for i in range(Xi_test_pca.shape[1])]
)

# 임상 + PCA 이미지 합치기
X_train_pca = pd.concat([Xc_train, Xi_train_pca_df], axis=1)
X_test_pca  = pd.concat([Xc_test,  Xi_test_pca_df],  axis=1)

In [44]:
# 컬럼 이름 string으로 통일
X_train_pca.columns = X_train_pca.columns.astype(str)
X_test_pca.columns  = X_test_pca.columns.astype(str)

cox_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("model", CoxPHSurvivalAnalysis(alpha=0.0))  # 혹은 위에서 찾은 best alpha 써도 됨
])

cox_pca.fit(X_train_pca, y_train)

cox_pca_train_c = cox_pca.score(X_train_pca, y_train)
cox_pca_test_c  = cox_pca.score(X_test_pca,  y_test)

print(f"Cox + PCA(50)  train C={cox_pca_train_c:.3f},  test C={cox_pca_test_c:.3f}")

Cox + PCA(50)  train C=1.000,  test C=0.667


/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sksurv/linear_model/coxph.py:200: RuntimeWarning: overflow encountered in exp
  risk_set += np.exp(xw[k])
/home/yuz/miniconda3/envs/pathML/lib/python3.10/site-packages/sklearn/pipeline.py:663: ConvergenceWarning: Optimization did not converge: Maximum number of iterations has been exceeded.
  self._final_estimator.fit(Xt, y, **last_step_params["fit"])


## 2. Random Survival Forest


In [37]:
rsf = RandomSurvivalForest(
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rsf.fit(X_train, y_train)

rsf_train_c = rsf.score(X_train, y_train)
rsf_test_c = rsf.score(X_test, y_test)

print(f"RSF     - train C-index: {rsf_train_c:.3f}, test C-index: {rsf_test_c:.3f}")

RSF     - train C-index: 0.970, test C-index: 0.529


## 3. Gradient Boosting Survival

In [38]:
gbsa = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GradientBoostingSurvivalAnalysis(random_state=42))
])

gbsa.fit(X_train, y_train)

gbsa_train_c = gbsa.score(X_train, y_train)
gbsa_test_c = gbsa.score(X_test, y_test)

print(f"GBSA    - train C-index: {gbsa_train_c:.3f}, test C-index: {gbsa_test_c:.3f}")

GBSA    - train C-index: 0.998, test C-index: 0.422


In [39]:
print("\n=== Summary (C-index) ===")
print(f"CoxPH   - train: {cox_train_c:.3f}, test: {cox_test_c:.3f}")
print(f"RSF     - train: {rsf_train_c:.3f}, test: {rsf_test_c:.3f}")
print(f"GBSA    - train: {gbsa_train_c:.3f}, test: {gbsa_test_c:.3f}")


=== Summary (C-index) ===
CoxPH   - train: 1.000, test: 0.667
RSF     - train: 0.970, test: 0.529
GBSA    - train: 0.998, test: 0.422


In [45]:
n_estimators_list = [50, 100, 200]
min_samples_leaf_list = [5, 10, 20]

rsf_results = []

for n_tree in n_estimators_list:
    for leaf in min_samples_leaf_list:
        rsf = RandomSurvivalForest(
            n_estimators=n_tree,
            min_samples_split=10,
            min_samples_leaf=leaf,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        )
        rsf.fit(X_train, y_train)
        train_c = rsf.score(X_train, y_train)
        test_c = rsf.score(X_test, y_test)
        rsf_results.append((n_tree, leaf, train_c, test_c))
        print(f"n_estimators={n_tree:3}, min_leaf={leaf:2}  "
              f"train C={train_c:.3f}, test C={test_c:.3f}")

n_estimators= 50, min_leaf= 5  train C=0.978, test C=0.490
n_estimators= 50, min_leaf=10  train C=0.970, test C=0.569
n_estimators= 50, min_leaf=20  train C=0.964, test C=0.431
n_estimators=100, min_leaf= 5  train C=0.981, test C=0.451
n_estimators=100, min_leaf=10  train C=0.978, test C=0.549
n_estimators=100, min_leaf=20  train C=0.962, test C=0.471
n_estimators=200, min_leaf= 5  train C=0.983, test C=0.412
n_estimators=200, min_leaf=10  train C=0.976, test C=0.451
n_estimators=200, min_leaf=20  train C=0.959, test C=0.412
